In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Setup — create directories and copy data from Person 2's dataset

In [ ]:
import os, shutil, glob

for d in ["data", "model", "train", "eval"]:
    os.makedirs(d, exist_ok=True)

for d in ["model", "train", "eval"]:
    open(f"{d}/__init__.py", "w").close()

for name in ["chunks.json", "train_pairs.json", "val_eval.json", "test_eval.json"]:
    src = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if src:
        shutil.copy(src[0], f"data/{name}")
        print(f"copied data/{name}")

src = glob.glob('/kaggle/input/**/tokenizer.json', recursive=True)
if src:
    shutil.copy(src[0], "tokenizer.json")
    print("copied tokenizer.json")

src = glob.glob('/kaggle/input/**/metrics.py', recursive=True)
if src:
    shutil.copy(src[0], "eval/metrics.py")
    print("copied eval/metrics.py")

config.py

In [ ]:
%%writefile config.py
import os
import torch


def _int(name, default):
    return int(os.environ.get(name, default))


def _flt(name, default):
    return float(os.environ.get(name, default))


SEED = _int("NS_SEED", 42)

DATA_DIR = os.environ.get("NS_DATA_DIR", "data")
BOOK_PATH = os.path.join(DATA_DIR, "book.txt")
CHUNKS_PATH = os.path.join(DATA_DIR, "chunks.json")
CORPUS_PATH = os.path.join(DATA_DIR, "book.txt")
TRAIN_PAIRS_PATH = os.path.join(DATA_DIR, "train_pairs.json")
VAL_EVAL_PATH = os.path.join(DATA_DIR, "val_eval.json")
TEST_EVAL_PATH = os.path.join(DATA_DIR, "test_eval.json")
TOKENIZER_PATH = os.environ.get("NS_TOKENIZER", "tokenizer.json")
MODEL_PATH = os.environ.get("NS_MODEL", "best_model.pt")

CHUNK_MIN = _int("NS_CHUNK_MIN", 200)
CHUNK_MAX = _int("NS_CHUNK_MAX", 300)

VOCAB_SIZE = _int("NS_VOCAB_SIZE", 8000)

D_MODEL = _int("NS_D_MODEL", 256)
NHEAD = _int("NS_NHEAD", 4)
NUM_LAYERS = _int("NS_NUM_LAYERS", 4)
DIM_FF = _int("NS_DIM_FF", 512)
DROPOUT = _flt("NS_DROPOUT", 0.1)
MAX_LEN = _int("NS_MAX_LEN", 320)
MODEL_MAX_LEN = _int("NS_MODEL_MAX_LEN", 512)

PRETRAIN_EPOCHS = _int("NS_PRETRAIN_EPOCHS", 3)
PRETRAIN_BS = _int("NS_PRETRAIN_BS", 64)
PRETRAIN_LR = _flt("NS_PRETRAIN_LR", 1e-4)
FINETUNE_EPOCHS = _int("NS_FINETUNE_EPOCHS", 50)
FINETUNE_BS = _int("NS_FINETUNE_BS", 32)
FINETUNE_LR = _flt("NS_FINETUNE_LR", 1e-4)
TEMPERATURE = _flt("NS_TEMPERATURE", 0.05)

EVAL_K = _int("NS_EVAL_K", 10)
PAIRS_PER_CHUNK = _int("NS_PAIRS_PER_CHUNK", 3)

DEVICE = os.environ.get("NS_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

model/model.py and model/contrastive_learning.py

In [ ]:
%%writefile model/model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def masked_mean(hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (hidden * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.1):
        super().__init__()
        assert d_model % nhead == 0
        self.nhead = nhead
        self.d_head = d_model // nhead
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask):
        B, L, d = x.shape
        def split(t):
            return t.view(B, L, self.nhead, self.d_head).transpose(1, 2)
        q, k, v = split(self.q_proj(x)), split(self.k_proj(x)), split(self.v_proj(x))
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if key_padding_mask is not None:
            scores = scores.masked_fill(key_padding_mask.view(B, 1, 1, L), float("-inf"))
        attention = self.dropout(torch.softmax(scores, dim=-1))
        ctx = (attention @ v).transpose(1, 2).contiguous().view(B, L, d)
        return self.out_proj(ctx)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, nhead, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask):
        x = x + self.dropout(self.attn(self.norm1(x), key_padding_mask))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x


class TransformerEncoderModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=4,
                 dim_feedforward=512, max_len=512, dropout=0.1, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos = PositionalEncoding(d_model, max_len)
        self.in_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, input_ids, attention_mask):
        x = self.embed(input_ids) * math.sqrt(self.d_model)
        x = self.in_dropout(self.pos(x))
        key_padding_mask = attention_mask == 0
        for block in self.blocks:
            x = block(x, key_padding_mask)
        x = self.final_norm(x)
        return masked_mean(x, attention_mask)


class BagOfEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model=256, pad_id=0, dropout=0.1):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask):
        return masked_mean(self.dropout(self.embed(input_ids)), attention_mask)

In [ ]:
%%writefile model/contrastive_learning.py
import torch
import torch.nn.functional as F


def info_nce(query_emb, passage_emb, temperature=0.05):
    q = F.normalize(query_emb, dim=-1)
    p = F.normalize(passage_emb, dim=-1)
    logits = (q @ p.t()) / temperature
    labels = torch.arange(q.size(0), device=q.device)
    loss_q = F.cross_entropy(logits, labels)
    loss_p = F.cross_entropy(logits.t(), labels)
    return 0.5 * (loss_q + loss_p)

build_tokenizer.py

In [ ]:
%%writefile build_tokenizer.py
from typing import Callable, List, Tuple

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

import config

SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]


def train_tokenizer(files: List[str], vocab_size: int = config.VOCAB_SIZE, out_path: str = config.TOKENIZER_PATH) -> Tokenizer:
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    tokenizer.decoder = decoders.BPEDecoder()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIAL_TOKENS)
    tokenizer.train(files, trainer)
    tokenizer.save(out_path)
    print(f"saved tokenizer -> {out_path}  (vocab={tokenizer.get_vocab_size()})")
    return tokenizer


def load_tokenizer(path: str = config.TOKENIZER_PATH) -> Tuple[Callable[[str], List[int]], int, int]:
    tokenizer = Tokenizer.from_file(path)
    pad_id = tokenizer.token_to_id("[PAD]")
    vocab_size = tokenizer.get_vocab_size()
    tokenize = lambda text: tokenizer.encode(text).ids
    return tokenize, pad_id, vocab_size


def main() -> None:
    train_tokenizer([config.BOOK_PATH])

data.py

In [ ]:
%%writefile data.py
from typing import Callable, List, Tuple

import torch
from torch import Tensor
from torch.utils.data import Dataset

import config


class PairDataset(Dataset):

    def __init__(self, pairs, tokenize: Callable[[str], List[int]], max_length: int = config.MAX_LEN) -> None:
        self.pairs = pairs
        self.tokenize = tokenize
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> Tuple[List[int], List[int]]:
        query, passage = self.pairs[index]
        return self.tokenize(query)[: self.max_length], self.tokenize(passage)[: self.max_length]


class TextDataset(Dataset):

    def __init__(self, texts, tokenize: Callable[[str], List[int]], max_length: int = config.MAX_LEN) -> None:
        self.texts = texts
        self.tokenize = tokenize
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int) -> List[int]:
        return self.tokenize(self.texts[index])[: self.max_length]


def pad_batch(sequences: List[List[int]], pad_id: int = 0) -> Tuple[Tensor, Tensor]:
    sequences = [seq if len(seq) > 0 else [pad_id] for seq in sequences]
    max_length = max(len(seq) for seq in sequences)
    input_ids = torch.full((len(sequences), max_length), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((len(sequences), max_length), dtype=torch.long)
    for i, seq in enumerate(sequences):
        input_ids[i, : len(seq)] = torch.tensor(seq, dtype=torch.long)
        attention_mask[i, : len(seq)] = 1
    return input_ids, attention_mask


def pair_collate(batch, pad_id: int = 0) -> Tuple[Tuple[Tensor, Tensor], Tuple[Tensor, Tensor]]:
    queries, passages = zip(*batch)
    return pad_batch(queries, pad_id), pad_batch(passages, pad_id)


def text_collate(batch, pad_id: int = 0) -> Tuple[Tensor, Tensor]:
    return pad_batch(batch, pad_id)

train/trainer.py

In [ ]:
%%writefile train/trainer.py
import json
import math
import os

import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

import config
from build_tokenizer import load_tokenizer
from data import PairDataset, pair_collate
from model.contrastive_learning import info_nce
from model.model import TransformerEncoderModel


def get_scheduler(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(step):
        if step < num_warmup_steps:
            return step / max(1, num_warmup_steps)
        progress = (step - num_warmup_steps) / max(1, num_training_steps - num_warmup_steps)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return LambdaLR(optimizer, lr_lambda)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total, n = 0.0, 0
    for (q_ids, q_mask), (p_ids, p_mask) in loader:
        q_ids, q_mask = q_ids.to(device), q_mask.to(device)
        p_ids, p_mask = p_ids.to(device), p_mask.to(device)
        loss = info_nce(model(q_ids, q_mask), model(p_ids, p_mask), config.TEMPERATURE)
        total += loss.item()
        n += 1
    model.train()
    return total / max(1, n)


def train(model, train_loader, val_loader, optimizer, scheduler, scaler, writer, epochs, device):
    best_val_loss = float("inf")
    step = 0

    for epoch in range(epochs):
        model.train()
        for (q_ids, q_mask), (p_ids, p_mask) in train_loader:
            q_ids, q_mask = q_ids.to(device), q_mask.to(device)
            p_ids, p_mask = p_ids.to(device), p_mask.to(device)

            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                loss = info_nce(model(q_ids, q_mask), model(p_ids, p_mask), config.TEMPERATURE)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            writer.add_scalar("train/loss", loss.item(), step)
            writer.add_scalar("train/lr", scheduler.get_last_lr()[0], step)
            step += 1

        val_loss = evaluate(model, val_loader, device)
        writer.add_scalar("val/loss", val_loss, epoch)
        print(f"epoch {epoch + 1}/{epochs}  val_loss={val_loss:.4f}")

        torch.save(model.state_dict(), f"checkpoint_epoch{epoch + 1}.pt")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), config.MODEL_PATH)
            print(f"  -> best model saved (val_loss={val_loss:.4f})")

    return best_val_loss


def main():
    torch.manual_seed(config.SEED)
    device = torch.device(config.DEVICE)

    tokenize, pad_id, vocab_size = load_tokenizer()

    with open(config.TRAIN_PAIRS_PATH, encoding="utf-8") as f:
        train_pairs = json.load(f)

    with open(config.VAL_EVAL_PATH, encoding="utf-8") as f:
        val_data = json.load(f)
    with open(config.CHUNKS_PATH, encoding="utf-8") as f:
        chunks = json.load(f)

    chunk_lookup = {c["id"]: c["text"] for c in chunks}
    val_pairs = [[q, chunk_lookup[g]] for q, g in zip(val_data["queries"], val_data["gold"])]

    def collate(batch):
        return pair_collate(batch, pad_id)

    train_loader = DataLoader(
        PairDataset(train_pairs, tokenize),
        batch_size=config.FINETUNE_BS,
        shuffle=True,
        collate_fn=collate,
    )
    val_loader = DataLoader(
        PairDataset(val_pairs, tokenize),
        batch_size=config.FINETUNE_BS,
        shuffle=False,
        collate_fn=collate,
    )

    model = TransformerEncoderModel(
        vocab_size=vocab_size,
        d_model=config.D_MODEL,
        nhead=config.NHEAD,
        num_layers=config.NUM_LAYERS,
        dim_feedforward=config.DIM_FF,
        max_len=config.MODEL_MAX_LEN,
        dropout=config.DROPOUT,
        pad_id=pad_id,
    ).to(device)

    num_training_steps = config.FINETUNE_EPOCHS * len(train_loader)
    num_warmup_steps = num_training_steps // 10

    optimizer = AdamW(model.parameters(), lr=config.FINETUNE_LR)
    scheduler = get_scheduler(optimizer, num_warmup_steps, num_training_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    writer = SummaryWriter("runs/training")

    print(f"device={device} | train={len(train_pairs)} pairs | val={len(val_pairs)} pairs")
    train(model, train_loader, val_loader, optimizer, scheduler, scaler, writer, config.FINETUNE_EPOCHS, device)
    writer.close()
    print("training complete.")


if __name__ == "__main__":
    main()

eval/vector_search.py

In [ ]:
%%writefile eval/vector_search.py
import json

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import config
from build_tokenizer import load_tokenizer
from data import TextDataset, text_collate
from model.model import TransformerEncoderModel


class VectorSearch:
    def __init__(self, model, tokenize, pad_id, chunks, device):
        self.model = model
        self.tokenize = tokenize
        self.pad_id = pad_id
        self.chunks = chunks
        self.device = device
        self.index = None

    def build_index(self, batch_size=64):
        texts = [c["text"] for c in self.chunks]
        dataset = TextDataset(texts, self.tokenize)
        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            collate_fn=lambda b: text_collate(b, self.pad_id),
        )
        embeddings = []
        self.model.eval()
        with torch.no_grad():
            for input_ids, attention_mask in loader:
                emb = self.model(input_ids.to(self.device), attention_mask.to(self.device))
                embeddings.append(emb.cpu())
        self.index = F.normalize(torch.cat(embeddings, dim=0), dim=-1)
        print(f"index built: {self.index.shape[0]} chunks x {self.index.shape[1]} dims")

    def retrieve(self, query: str, k: int) -> list:
        assert self.index is not None, "call build_index() first"
        self.model.eval()
        with torch.no_grad():
            ids = torch.tensor(
                [self.tokenize(query)[: config.MAX_LEN]], dtype=torch.long
            ).to(self.device)
            mask = torch.ones_like(ids)
            q_emb = F.normalize(self.model(ids, mask), dim=-1).cpu()
        scores = (q_emb @ self.index.T)[0]
        top_indices = scores.topk(k).indices.tolist()
        return [self.chunks[i]["id"] for i in top_indices]


def load(
    model_path=config.MODEL_PATH,
    tokenizer_path=config.TOKENIZER_PATH,
    chunks_path=config.CHUNKS_PATH,
):
    tokenize, pad_id, vocab_size = load_tokenizer(tokenizer_path)
    device = torch.device(config.DEVICE)

    model = TransformerEncoderModel(
        vocab_size=vocab_size,
        d_model=config.D_MODEL,
        nhead=config.NHEAD,
        num_layers=config.NUM_LAYERS,
        dim_feedforward=config.DIM_FF,
        max_len=config.MODEL_MAX_LEN,
        dropout=config.DROPOUT,
        pad_id=pad_id,
    ).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))

    with open(chunks_path, encoding="utf-8") as f:
        chunks = json.load(f)

    vs = VectorSearch(model, tokenize, pad_id, chunks, device)
    vs.build_index()
    return vs

## Run Training

In [ ]:
!python -m train.trainer

## Loss Curves

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import matplotlib.pyplot as plt

ea = EventAccumulator("runs/training")
ea.Reload()

train_loss = [(e.step, e.value) for e in ea.Scalars("train/loss")]
val_loss   = [(e.step, e.value) for e in ea.Scalars("val/loss")]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

steps, values = zip(*train_loss)
ax1.plot(steps, values)
ax1.set_title("Train Loss")
ax1.set_xlabel("step")
ax1.set_ylabel("loss")

epochs, values = zip(*val_loss)
ax2.plot(epochs, values, marker="o", color="orange")
ax2.set_title("Val Loss")
ax2.set_xlabel("epoch")
ax2.set_ylabel("loss")

plt.tight_layout()
plt.savefig("loss_curves.png", dpi=150)
plt.show()
print("saved loss_curves.png")

In [ ]:
import zipfile, glob

with zipfile.ZipFile('training_output.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write('best_model.pt')
    zf.write('loss_curves.png')
    for f in glob.glob('checkpoint_epoch*.pt'):
        zf.write(f)
    for f in glob.glob('runs/**/*', recursive=True):
        zf.write(f)

print("training_output.zip ready")